In [0]:
%run ./00_config

In [0]:
%run ./00_functions

In [0]:
import pyspark.sql.functions as F

In [0]:
# Read the bronze table stock in the database bakehouse_training
bronze_transactions_df = spark.read.table(bronze_transactions_fact)
bronze_customers_df = spark.read.table(bronze_customers_dim)
bronze_franchises_df = spark.read.table(bronze_franchises_dim)

In [0]:
# Standarization 
std_transactions_fact = standarize_transactions(bronze_transactions_df)
std_customers_dim = standarize_customers(bronze_customers_df)
std_franchises_dim = standarize_franchises(bronze_franchises_df)

In [0]:
std_transactions_fact.show(5)
std_customers_dim.show(5)
std_franchises_dim.show(5)

In [0]:
# Deduplication
dedup_transactions_fact, quality_df_transactions = check_global_quality(std_transactions_fact, 'transactionID')
dedup_customers_dim, quality_df_customers = check_global_quality(std_customers_dim, 'customerID')
dedup_franchises_dim, quality_df_franchises = check_global_quality(std_franchises_dim, 'franchiseID')

In [0]:
# Validation
checked_customers_dim = validate_customers(dedup_customers_dim)
checked_franchises_dim = validate_franchises(dedup_franchises_dim)
checked_transactions_fact = validate_transactions(dedup_transactions_fact, checked_customers_dim, checked_franchises_dim)

In [0]:
checked_franchises_dim.show(5)

In [0]:
# Split valid and invalid rows into 2 differents tables 
valid_transactions_fact, invalid_transactions_fact = split_valid_invalid(checked_transactions_fact)
valid_customers_dim, invalid_customers_dim = split_valid_invalid(checked_customers_dim)
valid_franchises_dim, invalid_franchises_dim = split_valid_invalid(checked_franchises_dim)

In [0]:
# Write the silver valid table stock in the database bakehouse_training
write_silver(valid_transactions_fact, silver_transactions_fact)
write_silver(valid_customers_dim, silver_customers_dim)
write_silver(valid_franchises_dim, silver_franchises_dim)
# The reject tables 
write_silver(invalid_transactions_fact,silver_invalid_transactions_fact)
write_silver(invalid_customers_dim, silver_invalid_customers_dim)
write_silver(invalid_franchises_dim, silver_invalid_franchises_dim)